# Buscas em memória interna: qual método usar em um catálogo de pedidos?
    **SSC0503 • Sistemas de Informação • ICMC–USP**

    **Trabalho individual • prazo: 10 dias a partir da disponibilização pelo professor.**

    Nome: ____________________  |  Número USP: ____________________

    Você implementará em **C11** os métodos de busca estudados e fará três investigações experimentais. Python já está preparado para gerar dados, compilar, testar e visualizar resultados.

    **Entregue este notebook preenchido, com todas as células executadas, gráficos e respostas.** Em Colab, use Arquivo → Fazer download → Fazer download do .ipynb. Salve suas alterações antes de fechar. Os arquivos temporários do ambiente podem desaparecer; os códigos escritos nas células permitem recriá-los.


## Objetivos, escopo e avaliação
    Uma empresa mantém identificadores de pedidos em memória e precisa consultá-los. O catálogo será um vetor de inteiros ordenado em ordem não decrescente. Os dados são sintéticos; representam cenários de espaçamento regular e grandes lacunas.

    Você deverá: implementar busca sequencial indexada, por salto, por interpolação e exponencial; recuperar busca sequencial e binária da aula anterior; validar limites e chaves ausentes; explicar os resultados.

    | Critério | Peso |
    |---|---:|
    | Correção dos quatro métodos principais (12,5% cada) | 50% |
    | Limites, casos especiais e contrato de instrumentação | 15% |
    | Clareza, funções auxiliares, revisão sequencial/binária e memória | 15% |
    | Análise experimental e conclusões | 20% |

    Planejamento sugerido: dias 1–2, interface e revisão; dias 3–6, quatro métodos; dias 7–8, testes e experimentos; dias 9–10, respostas e revisão. Estimativa: 8–12 horas, dependendo da familiaridade com C.

    Use os pseudocódigos como orientação. Não use `bsearch`, chamadas a implementações prontas de busca, C++ ou buscas em Python para substituir as implementações C. Bibliotecas padrão de C para entrada/saída e memória são permitidas. Explique com suas palavras as decisões tomadas.


## 1. Ambiente e fluxo de trabalho
    Abra o arquivo no Google Colab e selecione um ambiente **CPU**. Execute as células em ordem. A infraestrutura utiliza GCC, NumPy, pandas e Matplotlib. Se GCC não estiver disponível, troque para um ambiente Colab padrão.

    Cada célula C vazia produzirá um arquivo mediante a diretiva `%%writefile nome_do_arquivo` (digite essa diretiva antes do código). Ela é uma instrução do notebook, não uma instrução C. Todo o conteúdo C é escrito por você.

    Primeiro complete os arquivos; depois compile e valide; por fim execute os experimentos. Após mudar uma implementação, execute sua célula, compile novamente e repita os testes. A primeira compilação exige todas as funções; para desenvolvimento parcial, você pode criar temporariamente funções que retornam “não encontrado”, mas os testes correspondentes falharão até completar os métodos.


In [ ]:
import io, math, pathlib, shutil, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
pd.set_option('display.max_columns', 12)
METODOS = ['sequencial', 'binaria', 'indexada', 'salto', 'interpolacao', 'exponencial']
print('GCC:', shutil.which('gcc') or 'Não encontrado: use um ambiente Colab com CPU.')
print('Python pronto. As células C vazias devem ser preenchidas pelo aluno.')


## 2. Contrato comum e cabeçalho
    Todas as posições são **iniciadas em zero**. Retorne uma posição cuja chave seja igual à consulta; retorne `-1` se ela não existe. Com duplicatas, qualquer ocorrência é aceita. Os intervalos da busca binária são **fechados**: `[lo, hi]`; `lo > hi` representa intervalo vazio.

    Para esta atividade: `0 ≤ n ≤ 1.000.000`; bloco entre 1 e 1.000.000; chaves entre −10¹² e 10¹². O programa de testes fornecerá entradas válidas e ordenadas. Esses limites permitem índices `int` e chaves `long long`; a interpolação ainda exige conversão antes das operações para evitar overflow na multiplicação.

    **Métrica obrigatória:** uma chamada a `comparar` representa uma comparação tricotômica entre duas chaves e incrementa o contador uma única vez. O resultado é −1, 0 ou 1. Esse modelo é uma operação abstrata, não a contagem das instruções relacionais de C. Todas as decisões que comparam chaves, inclusive limites e extremos na interpolação, devem usar essa função. Comparações de índices não contam; o contador inicia em zero para cada consulta. Construção do índice não entra na contagem da consulta.

    Crie `buscas.h` com proteção contra inclusão repetida e as declarações abaixo. Esta estrutura e estas assinaturas são fornecidas apenas aqui, em texto, para padronizar a integração. Inclua o cabeçalho nos arquivos C.


```c
#ifndef BUSCAS_H
#define BUSCAS_H
#include <stddef.h>
typedef struct { long long maximo; int inicio; int quantidade; } Bloco;
int comparar(long long a, long long b, unsigned long long *c);
int sequencial(const long long *a, int n, long long x, unsigned long long *c);
int binaria(const long long *a, int lo, int hi, long long x, unsigned long long *c);
int construir_indice(const long long *a, int n, int b, Bloco **indice);
int indexada(const long long *a, const Bloco *indice, int m, long long x, unsigned long long *c);
int salto(const long long *a, int n, int b, long long x, unsigned long long *c);
int interpolacao(const long long *a, int n, long long x, unsigned long long *c);
int exponencial(const long long *a, int n, long long x, unsigned long long *c);
#endif
```


Em `Bloco`, `maximo` é a última chave do bloco; `inicio` é sua posição inicial; `quantidade` permite tratar o último bloco incompleto. `construir_indice` aloca o vetor de blocos e retorna sua quantidade; para vetor vazio, devolve zero e índice nulo; para falha de alocação, devolve −1. `main` libera esse vetor. O contador `c` é compartilhado pelas funções chamadas na mesma consulta; não o reinicialize dentro delas.


**Sua implementação — `buscas.h`.** Na célula vazia abaixo, digite `%%writefile buscas.h` na primeira linha e escreva todo o conteúdo C a seguir. Execute a célula para salvar o arquivo. Não use C++.


## 3. Revisão: sequencial, binária e comparador
    Implemente as três funções em `basicas.c`. A busca sequencial desta referência percorre todo o vetor quando não encontra a chave, sem encerramento antecipado por ordenação.

    ```text
    COMPARAR(a, b, contador):
        incrementar contador
        se a < b: devolver −1
        se a > b: devolver 1
        devolver 0

    SEQUENCIAL(A, n, x, contador):
        para i de 0 até n−1:
            se COMPARAR(A[i], x, contador) = 0: devolver i
        devolver −1

    BINÁRIA(A, lo, hi, x, contador):
        enquanto lo ≤ hi:
            meio ← lo + piso((hi−lo)/2)
            r ← COMPARAR(A[meio], x, contador)
            se r = 0: devolver meio
            se r < 0: lo ← meio+1
            senão: hi ← meio−1
        devolver −1
    ```

    Verificação manual: para `[2,4,6]`, a sequencial gasta 1, 3 e 3 comparações nas consultas 2, 6 e 7; a binária gasta uma na consulta 4. Explique por que a binária não precisa de acesso ao vetor quando `lo > hi`.


**Sua implementação — `basicas.c`.** Na célula vazia abaixo, digite `%%writefile basicas.c` na primeira linha e escreva todo o conteúdo C a seguir. Execute a célula para salvar o arquivo. Não use C++.


## 4. Exercício A — Busca sequencial indexada
    Implemente `construir_indice` e `indexada` em `indexada.c`. O índice será consultado **sequencialmente**, para estudar o custo de duas varreduras: índice e bloco. Não substitua essa etapa por busca binária.

    ```text
    CONSTRUIR_ÍNDICE(A, n, b):
        se n = 0: devolver índice vazio, quantidade 0
        m ← teto(n/b)
        alocar m entradas; sinalizar falha se não for possível
        para j de 0 até m−1:
            início ← j*b
            quantidade ← mínimo(b, n−início)
            máximo ← A[início+quantidade−1]
            guardar início, quantidade e máximo na entrada j
        devolver índice e m

    INDEXADA(A, índice, m, x, contador):
        para j de 0 até m−1:
            se COMPARAR(índice[j].máximo, x, contador) ≥ 0:
                percorrer as posições do bloco j em ordem:
                    r ← COMPARAR(A[i], x, contador)
                    se r = 0: devolver i
                    se r > 0: devolver −1
                devolver −1
        devolver −1
    ```

    Exemplo: `[10,20,30,40,50,60,70]`, bloco 3 → máximos `[30,60,70]`; a consulta 50 seleciona o segundo bloco e percorre 40, 50. São quatro comparações neste modelo.

    **Questões A:** A1. Justifique a escolha do primeiro máximo ≥ chave, inclusive com duplicatas. A2. Derive `O(n/b+b)` e explique a escolha próxima de √n. A3. Informe o custo da construção e o espaço do índice; diferencie o processamento desta representação da leitura prévia de todos os dados.


**Sua implementação — `indexada.c`.** Na célula vazia abaixo, digite `%%writefile indexada.c` na primeira linha e escreva todo o conteúdo C a seguir. Execute a célula para salvar o arquivo. Não use C++.


**Respostas A:**

_Escreva aqui suas respostas A1–A3._


## 5. Exercício B — Busca por salto
    Implemente `salto` em `salto.c`. Use o mesmo tamanho de bloco da indexada, mas consulte diretamente as extremidades do vetor, sem armazenar o índice.

    ```text
    SALTO(A, n, b, x, contador):
        para início em 0, b, 2b, ... enquanto início < n:
            fim ← mínimo(início+b−1, n−1)
            se COMPARAR(A[fim], x, contador) ≥ 0:
                para i de início até fim:
                    r ← COMPARAR(A[i], x, contador)
                    se r = 0: devolver i
                    se r > 0: devolver −1
                devolver −1
        devolver −1
    ```

    **Questões B:** B1. Compare a memória adicional com a indexada. B2. Para estes dois pseudocódigos, as contagens de consulta deveriam diferir? Justifique. B3. Explique como lidar com `b > n` e com o último bloco incompleto.


**Sua implementação — `salto.c`.** Na célula vazia abaixo, digite `%%writefile salto.c` na primeira linha e escreva todo o conteúdo C a seguir. Execute a célula para salvar o arquivo. Não use C++.


**Respostas B:**

_Escreva aqui suas respostas B1–B3._


## 6. Exercício C — Busca por interpolação
    Implemente `interpolacao` em `interpolacao.c`. A estimativa usa valores das chaves, além de posições.

    ```text
    INTERPOLAÇÃO(A, n, x, contador):
        lo ← 0; hi ← n−1
        enquanto lo ≤ hi:
            se COMPARAR(x, A[lo], contador) < 0: devolver −1
            se COMPARAR(x, A[hi], contador) > 0: devolver −1
            se COMPARAR(A[lo], A[hi], contador) = 0:
                se COMPARAR(x, A[lo], contador) = 0: devolver lo
                devolver −1
            razão ← (x−A[lo]) / (A[hi]−A[lo]), em ponto flutuante
            pos ← lo + piso(razão*(hi−lo))
            limitar pos ao intervalo [lo,hi] por segurança numérica
            r ← COMPARAR(A[pos], x, contador)
            se r = 0: devolver pos
            se r < 0: lo ← pos+1
            senão: hi ← pos−1
        devolver −1
    ```

    Em C, converta os operandos para `long double` **antes** de subtrair e multiplicar na estimativa. Não faça divisão inteira. Como a razão é não negativa, a conversão da estimativa para inteiro corresponde ao piso. Os limites do enunciado não abrangem todo o domínio de `long long`.

    **Questões C:** C1. Por que tratar extremos iguais antes de calcular a razão? C2. Por que `pos+1` e `pos−1` garantem progresso? C3. Diferencie o resultado médio clássico `O(log log n)` sob hipóteses adequadas de distribuição uniforme do pior caso `O(n)`. Os dados regularmente espaçados do experimento não são uma amostra aleatória uniforme: não use esse experimento como prova do teorema.


**Sua implementação — `interpolacao.c`.** Na célula vazia abaixo, digite `%%writefile interpolacao.c` na primeira linha e escreva todo o conteúdo C a seguir. Execute a célula para salvar o arquivo. Não use C++.


**Respostas C:**

_Escreva aqui suas respostas C1–C3._


## 7. Exercício D — Busca exponencial
    Implemente `exponencial` em `exponencial.c`, reutilizando sua função binária e seu contador. O tamanho é conhecido nesta implementação; verifique os limites antes de acessar o vetor.

    ```text
    EXPONENCIAL(A, n, x, contador):
        se n = 0: devolver −1
        r ← COMPARAR(A[0], x, contador)
        se r = 0: devolver 0
        se r > 0: devolver −1
        k ← 1
        enquanto k < n:
            se COMPARAR(A[k], x, contador) ≥ 0: encerrar expansão
            k ← 2*k
        devolver BINÁRIA(A, piso(k/2), mínimo(k,n−1), x, contador)
    ```

    O limite `n ≤ 1.000.000` torna a duplicação de `k` segura em `int`. Uma biblioteca genérica para vetores maiores precisaria proteger a duplicação contra overflow.

    **Questões D:** D1. Justifique por que o intervalo final contém a chave, se ela existe. D2. Expresse o custo em função da posição `i` como `O(log(i+1))`, considerando a consulta na posição zero. Para ausência, discuta a posição de inserção. D3. Como adaptar conceitualmente o método a uma sequência de tamanho desconhecido sem acessar memória inválida?


**Sua implementação — `exponencial.c`.** Na célula vazia abaixo, digite `%%writefile exponencial.c` na primeira linha e escreva todo o conteúdo C a seguir. Execute a célula para salvar o arquivo. Não use C++.


**Respostas D:**

_Escreva aqui suas respostas D1–D3._


## 8. Programa principal e protocolo de integração
    Escreva `main.c`. Ele deverá aceitar dois argumentos: nome do método e tamanho do bloco. Métodos válidos: `sequencial`, `binaria`, `indexada`, `salto`, `interpolacao`, `exponencial`.

    A entrada padrão contém: primeira linha com `n q`; depois `n` identificadores; depois `q` consultas. Os espaços e as quebras de linha são separadores. A saída padrão deve conter **somente CSV**, sem mensagens de depuração. Use a saída de erro para mensagens de diagnóstico.

    Exemplo de entrada:
    ```text
    3 2
    10 20 30
    20 99
    ```
    Saída esperada usando o método sequencial:
    ```text
    consulta,chave,posicao,comparacoes
    0,20,1,2
    1,99,-1,3
    ```

    ```text
    PRINCIPAL:
        verificar argumentos; interpretar tamanho do bloco
        ler n e q; alocar vetor quando n > 0; ler identificadores
        se método indexada: construir índice uma única vez
        escrever cabeçalho CSV
        para j de 0 até q−1:
            ler chave; contador ← 0
            chamar método selecionado (binária usa [0,n−1])
            escrever j, chave, posição e contador separados por vírgulas
        liberar vetor e índice; encerrar com código zero
    ```

    Inclua os cabeçalhos padrão necessários. Sugestões: `scanf`/`printf` para o protocolo, `malloc`/`free` para memória, `strcmp` para o método. Para chave `long long` use `%lld`; para contador `unsigned long long`, `%llu`. Verifique falhas de leitura e alocação. Não reconstrua o índice a cada consulta. Não ordene os dados novamente: a entrada já está ordenada.


**Sua implementação — `main.c`.** Na célula vazia abaixo, digite `%%writefile main.c` na primeira linha e escreva todo o conteúdo C a seguir. Execute a célula para salvar o arquivo. Não use C++.


## 9. Compilação e execução pelo Python
    A célula abaixo contém somente infraestrutura Python. Ela compila todos os arquivos em C11, mostra os avisos e disponibiliza uma função que valida a saída. Se houver falha, corrija o código C antes de seguir. O validador usa um conjunto de chaves como referência de presença, sem reproduzir os algoritmos pedidos.


In [ ]:
ARQUIVOS = ['buscas.h','basicas.c','indexada.c','salto.c','interpolacao.c','exponencial.c','main.c']
def compilar():
    faltam = [f for f in ARQUIVOS if not pathlib.Path(f).is_file()]
    if faltam:
        print('Ainda faltam arquivos:', ', '.join(faltam))
        print('Preencha e execute as células C. Veja a instrução %%writefile nas células de texto.')
        return False
    # Remove o executável antigo para que uma compilação falha não use código desatualizado.
    pathlib.Path('buscas').unlink(missing_ok=True)
    comando = ['gcc','-std=c11','-Wall','-Wextra','-Wpedantic','-O2'] + ARQUIVOS[1:] + ['-o','buscas']
    r = subprocess.run(comando, capture_output=True, text=True, timeout=30)
    if r.stdout: print(r.stdout)
    if r.stderr: print(r.stderr)
    print('Compilação concluída.' if r.returncode == 0 else 'Corrija os erros e compile novamente.')
    return r.returncode == 0
COMPILADO = compilar()

def executar(a, consultas, metodo, bloco=None):
    if not COMPILADO:
        raise RuntimeError('Implemente os arquivos e execute novamente a célula de compilação.')
    a = [int(x) for x in a]; consultas = [int(x) for x in consultas]
    assert a == sorted(a), 'O catálogo deve estar ordenado.'
    b = max(1, math.isqrt(len(a))) if bloco is None else int(bloco)
    entrada = f'{len(a)} {len(consultas)}\n' + ' '.join(map(str,a)) + '\n' + ' '.join(map(str,consultas)) + '\n'
    try:
        r = subprocess.run([str(pathlib.Path('buscas').resolve()), metodo, str(b)],
                           input=entrada, capture_output=True, text=True, timeout=20)
    except subprocess.TimeoutExpired as e:
        raise RuntimeError(f'{metodo}: execução excedeu 20 s. Verifique progresso e limites.') from e
    if r.returncode:
        raise RuntimeError(f'{metodo}: código de saída {r.returncode}. {r.stderr}')
    try:
        df = pd.read_csv(io.StringIO(r.stdout))
    except Exception as e:
        raise ValueError('Saída inválida: use exatamente o CSV especificado.') from e
    assert list(df.columns) == ['consulta','chave','posicao','comparacoes'], 'Cabeçalho incorreto.'
    assert len(df) == len(consultas), 'Quantidade incorreta de resultados.'
    existentes = set(a)
    for j, linha in enumerate(df.itertuples(index=False)):
        assert linha.consulta == j and linha.chave == consultas[j], 'Consulta fora de ordem.'
        assert linha.posicao == int(linha.posicao), 'Posição deve ser inteira.'
        p = int(linha.posicao)
        correto = (p == -1 and consultas[j] not in existentes) or (0 <= p < len(a) and a[p] == consultas[j])
        assert correto, f'{metodo}: chave {consultas[j]}, posição {p}; resultado incorreto.'
        assert linha.comparacoes >= 0 and linha.comparacoes == int(linha.comparacoes), 'Contagem inválida.'
    return df


## 10. Testes de correção
    Os testes cobrem vazio, um elemento, limites, ausência, duplicatas, valores iguais, negativos, lacunas grandes e vetores aleatórios. Blocos 1, 3 e maiores que n verificam limites. Os testes não substituem a leitura crítica do código: um número de comparações plausível não prova que a instrumentação é correta.

    Execute e acrescente pelo menos **dois casos próprios**, justificando que erro cada um pretende detectar. Você pode chamar `executar(vetor, consultas, metodo, bloco)` em uma célula Python adicional.


In [ ]:
def validar():
    casos = [([],[-1,0,1]), ([7],[6,7,8]), ([2,4,6,8],[1,2,3,8,9]),
             ([5]*9,[4,5,6]), ([1,1,2,2,2,9],[0,1,2,3,9,10]),
             ([-20,-5,0,10],[-21,-20,-6,-5,0,1,10,11]),
             ([0,1,2,3,10**12],[0,2,4,10**12-1,10**12,10**12+1])]
    rng = np.random.default_rng(503)
    for n in [2,3,11,57,200]:
        a = np.sort(rng.integers(-1000,1001,size=n)).tolist()
        qs = a[::max(1,n//10)] + rng.integers(-1002,1003,size=25).tolist()
        casos.append((a,qs))
    total = 0
    for metodo in METODOS:
        for a, qs in casos:
            for b in ([1,3,max(1,len(a)+2)] if metodo in ['indexada','salto'] else [1]):
                executar(a,qs,metodo,b)
                total += len(qs)
        print(metodo, '— casos de correção aprovados')
    # Verifica a instrumentação, não apenas a localização da chave.
    assert executar([2,4,6],[2,6,7],'sequencial')['comparacoes'].tolist() == [1,3,3]
    assert executar([2,4,6],[4],'binaria')['comparacoes'].tolist() == [1]
    print(f'{total} consultas verificadas; testes básicos de contagem aprovados.')
if COMPILADO:
    validar()
else:
    print('Testes aguardando suas implementações.')


**Justificativa dos casos próprios:**

_Descreva seus dois casos e os erros que eles detectariam._


## 11. Preparar os experimentos
    Serão usadas 120 consultas por cenário: metade presente e metade ausente. As ausentes têm valor uma unidade acima de chaves existentes; os catálogos gerados contêm somente múltiplos de dez. Os métodos recebem exatamente as mesmas consultas em cada cenário. A semente é fixa.

    Na consulta próxima do início, escolhemos posições no primeiro 1% do catálogo. Na distribuição com lacunas, 90% das chaves ocupam uma faixa curta e as demais ficam numa faixa distante. Isto é um cenário controlado, não uma base real.

    Não medimos tempo: as **comparações tricotômicas** são a métrica principal desta atividade. Elas não medem aritmética, cache ou tempo total; use essa limitação nas conclusões. Não há necessidade de GPU.


In [ ]:
RNG = np.random.default_rng(503)
def catalogo(n, distribuicao):
    if distribuicao == 'regular':
        return np.arange(n, dtype=np.int64) * 10 + 100
    if distribuicao == 'lacunas':
        # 90% dos pedidos se concentram numa faixa; os demais estão muito distantes.
        k = int(0.9*n)
        return np.concatenate([np.arange(k,dtype=np.int64)*10+100,
                               10**9+np.arange(n-k,dtype=np.int64)*10000])
    raise ValueError(distribuicao)

def consultas_catalogo(a, perfil, quantidade=120):
    limite = max(1,len(a)//100) if perfil == 'inicio' else len(a)
    ids = RNG.integers(0,limite,size=quantidade//2)
    presentes = a[ids].tolist()
    # Todas as chaves do catálogo terminam em zero; +1 garante ausência.
    ausentes = (a[ids]+1).tolist()
    registros = [(int(x),'presente') for x in presentes] + [(int(x),'ausente') for x in ausentes]
    RNG.shuffle(registros)
    return [x for x,t in registros], [t for x,t in registros]

def medir(a, qs, tipos, metodos=METODOS, bloco=None, **cenario):
    partes=[]
    for metodo in metodos:
        d=executar(a,qs,metodo,bloco)
        d['metodo']=metodo; d['tipo']=tipos
        d['n']=len(a); d['bloco']=max(1,math.isqrt(len(a))) if bloco is None else bloco
        for k,v in cenario.items(): d[k]=v
        partes.append(d)
    return pd.concat(partes,ignore_index=True)
print('Geradores prontos. Usaremos as mesmas consultas para os métodos de cada cenário.')


### Investigação A — O tamanho do bloco importa?
    Catálogo regular de 10.000 pedidos, mesmas consultas para todos os tamanhos. Execute a célula e responda: (1) qual região de tamanhos apresentou menor média? (2) por que essa região é compatível com √n sem garantir mínimo exato nessa amostra? (3) explique a coincidência ou diferença entre as curvas de salto e indexada; (4) se o bloco diminuir, o que acontece com a memória do índice?


In [ ]:
if COMPILADO:
    a=catalogo(10000,'regular')
    qs,tipos=consultas_catalogo(a,'espalhado')
    blocos=[1,5,10,25,50,100,200,500,1000,10000]
    e1=pd.concat([medir(a,qs,tipos,['indexada','salto'],b) for b in blocos],ignore_index=True)
    resumo1=e1.groupby(['metodo','bloco'])['comparacoes'].agg(['mean','median','max']).reset_index()
    display(resumo1.round(2))
    fig,ax=plt.subplots(figsize=(8,4))
    for metodo,g in resumo1.groupby('metodo'):
        ax.plot(g.bloco,g['mean'],marker='o',label=metodo)
    ax.axvline(math.isqrt(len(a)),color='gray',linestyle='--',label='sqrt(n)')
    ax.set(xscale='log',yscale='log',xlabel='Tamanho do bloco',ylabel='Comparações médias',title='Investigação A: tamanho do bloco')
    ax.legend(); ax.grid(alpha=.2); fig.tight_layout(); plt.show()
else: print('Experimento aguardando implementação e compilação.')


**Análise A:**

_Responda aos quatro itens com referência à sua tabela e ao gráfico._


### Investigação B — Interpolação funciona sempre melhor?
    Compare seis métodos em três tamanhos e duas distribuições. Separe consultas presentes e ausentes na tabela. Responda: (1) quando a interpolação foi favorável? (2) o que mudou com as lacunas? (3) compare binária e sequencial conforme n cresce; (4) por que os gráficos não provam complexidades assintóticas nem medem tempo físico?


In [ ]:
if COMPILADO:
    partes=[]
    for n in [100,1000,10000]:
        for distribuicao in ['regular','lacunas']:
            a=catalogo(n,distribuicao); qs,tipos=consultas_catalogo(a,'espalhado')
            partes.append(medir(a,qs,tipos,distribuicao=distribuicao))
    e2=pd.concat(partes,ignore_index=True)
    resumo2=e2.groupby(['distribuicao','n','metodo','tipo'])['comparacoes'].mean().reset_index()
    display(resumo2.pivot_table(index=['distribuicao','n','metodo'],columns='tipo',values='comparacoes').round(2))
    fig,axes=plt.subplots(1,2,figsize=(12,4))
    for ax,dist in zip(axes,['regular','lacunas']):
        t=e2[e2.distribuicao==dist].groupby(['n','metodo']).comparacoes.mean().reset_index()
        for metodo,g in t.groupby('metodo'): ax.plot(g.n,g.comparacoes,marker='o',label=metodo)
        ax.set(xscale='log',yscale='log',xlabel='Número de pedidos',ylabel='Comparações médias',title=dist)
        ax.grid(alpha=.2)
    axes[1].legend(fontsize=8); fig.suptitle('Investigação B: distribuição das chaves'); fig.tight_layout(); plt.show()
else: print('Experimento aguardando implementação e compilação.')


**Análise B:**

_Responda aos quatro itens com evidências dos resultados._


### Investigação C — Consultas perto do início
    Compare exponencial e binária em dados regulares. Responda: (1) em que cenários a exponencial usou menos comparações? (2) como a fase de expansão afeta consultas espalhadas? (3) por que uma chave ausente próxima do início também pode favorecer a exponencial? (4) o primeiro 1% sempre terá custo constante quando n aumenta?


In [ ]:
if COMPILADO:
    partes=[]
    for n in [100,1000,10000]:
        a=catalogo(n,'regular')
        for perfil in ['inicio','espalhado']:
            qs,tipos=consultas_catalogo(a,perfil)
            partes.append(medir(a,qs,tipos,['binaria','exponencial'],perfil=perfil))
    e3=pd.concat(partes,ignore_index=True)
    resumo3=e3.groupby(['perfil','n','metodo','tipo']).comparacoes.mean().reset_index()
    display(resumo3.pivot_table(index=['perfil','n','metodo'],columns='tipo',values='comparacoes').round(2))
    fig,ax=plt.subplots(figsize=(8,4))
    t=e3[e3.n==10000].groupby(['perfil','metodo']).comparacoes.mean().unstack()
    t.plot.bar(ax=ax,rot=0)
    ax.set(ylabel='Comparações médias',xlabel='Perfil das consultas',title='Investigação C: consultas no início (1%) ou espalhadas')
    fig.tight_layout(); plt.show()
else: print('Experimento aguardando implementação e compilação.')


**Análise C:**

_Responda aos quatro itens e cite os cenários observados._


## 12. Conclusão e entrega
    Em **250–400 palavras**, recomende métodos para: catálogo regular, catálogo com grandes lacunas e consultas concentradas no início. Inclua: pré-condições, custo de construção e memória do índice, métricas observadas, limitações do experimento e pelo menos uma situação em que você escolheria busca binária.

    Antes de entregar: preencha identificação; verifique que todo código é C; reinicie o ambiente e execute todas as células em ordem; confirme testes aprovados; revise gráficos e respostas; faça download do `.ipynb`. O notebook deve funcionar sem arquivos externos e sem acesso ao Drive. Se um teste falhar, registre e explique a falha, em vez de apagar o teste.


**Conclusão:**

_Escreva sua conclusão de 250–400 palavras._


## Referências e materiais de apoio
    - Knuth, D. E. **The Art of Computer Programming, Volume 3: Sorting and Searching**, 2ª ed., Addison-Wesley, 1998. Busca e análise de algoritmos.
    - Cormen, T. H. et al. **Introduction to Algorithms**, 4ª ed., MIT Press, 2022. Análise assintótica, invariantes e correção.
    - Perl, Y.; Itai, A.; Avni, H. **Interpolation Search—A Log LogN Search**. Communications of the ACM, 21(7), 550–553, 1978. DOI: [10.1145/359545.359557](https://doi.org/10.1145/359545.359557).
    - Bentley, J. L.; Yao, A. C. **An Almost Optimal Algorithm for Unbounded Searching**. Information Processing Letters, 5(3), 82–87, 1976. DOI: [10.1016/0020-0190(76)90071-5](https://doi.org/10.1016/0020-0190(76)90071-5).
    - GCC: [padrões de linguagem](https://gcc.gnu.org/onlinedocs/gcc/Standards.html).
    - IPython: [magia writefile](https://ipython.readthedocs.io/en/stable/interactive/magics.html#magic-writefile).
    - Google Colab: [perguntas frequentes](https://research.google.com/colaboratory/faq.html).

    Os pseudocódigos são adaptações didáticas para o contrato desta atividade. A busca por salto funciona aqui como comparação conceitual com o índice esparso.
